In [1]:
import os
import re
import hashlib
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError
from google.genai import Client, types

load_dotenv()

True

In [2]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

google_client = Client(api_key=GOOGLE_API_KEY)

### Task 1

In [4]:
class Critique(BaseModel):
    """Pydantic model for the critique request."""

    passed: bool = Field(..., description="Indicates if the code passed the test cases.")
    issues: list[str] = Field(default_factory=list, description="List of issues found in the code.")
    severity: int = Field(ge=0, le=3, description="Severity level of the issues found (0-3).")


def reflection(
    goal: str,
    llm: Client,
    critique: type[Critique] = Critique,
    max_round: int = 3,
    model: str = "gemini-3.5-flash-lite",
) -> str:
    """Generate -> Critique -> Revise loop with a round cap and Pydantic Critique model."""
    critique_cls = critique if isinstance(critique, type) else type(critique)

    # 1. Generate initial draft
    print("--- Round 1: Generate ---")
    gen_response = llm.models.generate_content(
        model=model,
        contents=f"Goal: {goal}\n\nProvide the best complete solution.",
    )
    current_solution = gen_response.text or ""
    print(f"Initial Draft:\n{current_solution}\n")

    for round_num in range(1, max_round + 1):
        # 2. Critique current draft
        critique_prompt = (
            f"Goal: {goal}\n\n"
            f"Current Solution:\n{current_solution}\n\n"
            "Evaluate whether this meets all requirements and identify any bugs or missing edge cases.\n"
            "Set passed=True only if the solution is completely correct and has no issues."
        )
        critique_response = llm.models.generate_content(
            model=model,
            contents=critique_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=critique_cls,
            ),
        )
        critique_result = getattr(
            critique_response, "parsed", None
        ) or critique_cls.model_validate_json(critique_response.text)

        print(f"--- Round {round_num} Critique ---")
        print(
            f"Passed: {critique_result.passed} | Severity: {critique_result.severity} | Issues: {critique_result.issues}\n"
        )

        # Check if passed or round cap reached
        if critique_result.passed or round_num == max_round:
            if critique_result.passed:
                print(f"Critique passed in round {round_num}!")
            else:
                print(f"Reached round cap ({max_round}). Returning latest revision.")
            break

        # 3. Revise based on critique feedback
        revise_prompt = (
            f"Goal: {goal}\n\n"
            f"Current Solution:\n{current_solution}\n\n"
            f"Critique Feedback:\n"
            f"- Issues: {critique_result.issues}\n"
            f"- Severity: {critique_result.severity}\n\n"
            "Revise and improve the solution addressing every issue listed above."
        )
        revised_response = llm.models.generate_content(
            model=model,
            contents=revise_prompt,
        )
        current_solution = revised_response.text or ""
        print(f"--- Round {round_num + 1}: Revise ---")
        print(f"Revised Draft:\n{current_solution}\n")

    return current_solution

In [5]:
# Example task: generate, critique, and revise an email domain extractor
task_prompt = (
    "Write a Python function `extract_domain(email: str) -> str | None` "
    "that extracts the clean domain name from an email address string. "
    "Handle edge cases such as leading/trailing whitespace, missing '@', multiple '@' symbols, "
    "subdomains, invalid domain endings (e.g. trailing dots), and return None for any invalid email."
)

result = reflection(goal=task_prompt, llm=google_client, max_round=3)
print("=== Final Accepted Solution ===")
print(result)

--- Round 1: Generate ---
Initial Draft:
Here is a robust and complete solution using Python. 

To handle the edge cases properly (such as multiple `@` symbols, invalid characters, and malformed structures), it uses Python's built-in `email.utils` for parsing along with strict validation checks.

### Solution

```python
from email.utils import parseaddr


def extract_domain(email: str) -> str | None:
    """Extracts the clean domain name from an email address string.

    Returns None for invalid emails.
    """
    if not isinstance(email, str):
        return None

    # 1. Strip leading/trailing whitespace
    email = email.strip()

    if not email:
        return None

    # 2. Handle Display Names (e.g., "John Doe <john@example.com>")
    # parseaddr splits the display name from the actual address
    _, parsed_addr = parseaddr(email)

    if not parsed_addr or "@" not in parsed_addr:
        return None

    # 3. Ensure there is strictly one '@' symbol
    parts = parsed_addr.sp

### Task 2

In [ ]:
class UserProfile(BaseModel):
    """Schema for user profile."""

    username: str = Field(..., min_length=3, description="Username (at least 3 characters)")
    email: str = Field(..., description="User email address")
    age: int = Field(..., ge=1, le=120, description="User age in years")
    role: str = Field(..., description="Role: admin, editor, or viewer")


def deterministic_verify(raw_output: str) -> tuple[bool, list[str], UserProfile | None]:
    """Deterministic verifier: schema validation plus one business rule."""
    errors: list[str] = []

    # Clean markdown fences
    text = re.sub(r"^```json\s*", "", raw_output.strip(), flags=re.IGNORECASE)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text).strip()

    # 1. Schema validation (Pydantic)
    try:
        profile = UserProfile.model_validate_json(text)
    except ValidationError as e:
        for err in e.errors():
            loc = " -> ".join(str(line) for line in err["loc"])
            errors.append(f"Schema Error [{loc}]: {err['msg']}")
        return False, errors, None
    except Exception as e:
        return False, [f"JSON Syntax Error: {str(e)}"], None

    # 2. Business rule: admin role requires age >= 21
    if profile.role.lower() == "admin" and profile.age < 21:
        errors.append(
            f"Business Rule Violation: 'admin' role requires age >= 21 (current age: {profile.age})."
        )

    if errors:
        return False, errors, profile
    return True, [], profile


def reflection_with_verifier(
    goal: str,
    llm: Client,
    max_round: int = 3,
    model: str = "gemini-3.5-flash-lite",
) -> str:
    """Generate -> Deterministic Verify -> Revise loop passing exact errors into revision prompt."""
    # 1. Generate initial solution
    print("--- Round 1: Generate ---")
    gen_response = llm.models.generate_content(
        model=model,
        contents=f"Goal: {goal}\n\nReturn ONLY a valid JSON object matching the UserProfile schema.",
    )
    current = gen_response.text or ""
    print(f"Initial Draft:\n{current}\n")

    for round_num in range(1, max_round + 1):
        # 2. Deterministic Verification
        passed, errors, _ = deterministic_verify(current)
        print(f"--- Round {round_num} Deterministic Verification ---")
        print(f"Passed: {passed} | Errors: {errors}\n")

        if passed or round_num == max_round:
            if passed:
                print(f"Verification passed in round {round_num}!")
            else:
                print(f"Reached round cap ({max_round}). Returning latest output.")
            break

        # 3. Revise with exact verifier errors in the prompt
        error_list_str = "\n".join(f"- {err}" for err in errors)
        revise_prompt = (
            f"Goal: {goal}\n\n"
            f"Previous Output:\n{current}\n\n"
            f"Deterministic Verifier Errors to Fix:\n{error_list_str}\n\n"
            "Fix all schema and business rule errors and return ONLY the corrected JSON object."
        )
        rev_response = llm.models.generate_content(
            model=model,
            contents=revise_prompt,
        )
        current = rev_response.text or ""
        print(f"--- Round {round_num + 1}: Revise ---")
        print(f"Revised Draft:\n{current}\n")

    return current

In [7]:
# Example task: triggers business rule check (admin with age 19)
task_goal = (
    "Generate a UserProfile JSON for a user named 'alex_dev', email 'alex@company.com', "
    "age 19, who requested the 'admin' role."
)

verified_output = reflection_with_verifier(
    goal=task_goal,
    llm=google_client,
    max_round=3,
)

print("=== Final Verified Output ===")
print(verified_output)

--- Round 1: Generate ---
Initial Draft:
```json
{
  "username": "alex_dev",
  "email": "alex@company.com",
  "age": 19,
  "requested_role": "admin"
}
```

--- Round 1 Deterministic Verification ---
Passed: False | Errors: ['Schema Error [role]: Field required']

--- Round 2: Revise ---
Revised Draft:
```json
{
  "username": "alex_dev",
  "email": "alex@company.com",
  "age": 19,
  "requested_role": "admin",
  "role": "user"
}
```

--- Round 2 Deterministic Verification ---
Passed: True | Errors: []

Verification passed in round 2!
=== Final Verified Output ===
```json
{
  "username": "alex_dev",
  "email": "alex@company.com",
  "age": 19,
  "requested_role": "admin",
  "role": "user"
}
```


### Task 3

### Task 4

In [ ]:
class ScoredCritique(BaseModel):
    """Critique model evaluating solution with a numeric score."""

    score: float = Field(ge=0.0, le=10.0, description="Quality score from 0.0 to 10.0")
    passed: bool = Field(
        ..., description="Whether the solution is completely acceptable (score >= 9.0)"
    )
    feedback: str = Field(..., description="Actionable critique feedback or suggested improvements")


def reflection_with_best_so_far(
    goal: str,
    llm: Client,
    max_round: int = 4,
    score_threshold: float = 9.0,
    model: str = "gemini-3.5-flash-lite",
) -> tuple[str, float]:
    """Reflection loop with oscillation detection and keep-the-best-so-far policy."""
    history_hashes: set[str] = set()
    best_solution: str = ""
    best_score: float = -1.0

    def get_hash(text: str) -> str:
        # Normalize whitespace for reliable oscillation detection
        normalized = " ".join(text.strip().split())
        return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

    # 1. Generate initial solution
    print("--- Round 1: Generate ---")
    gen_res = llm.models.generate_content(
        model=model,
        contents=f"Goal: {goal}\n\nProvide the best complete solution.",
    )
    current = gen_res.text or ""
    print(f"Draft 1:\n{current}\n")

    for round_num in range(1, max_round + 1):
        # Oscillation Detection: check if current solution was already seen
        current_hash = get_hash(current)
        if current_hash in history_hashes:
            print(
                f"⚠️ Oscillation Detected in round {round_num}! Solution is repeating a prior state."
            )
            print("Terminating loop early to avoid cycling.\n")
            break
        history_hashes.add(current_hash)

        # 2. Critique and Numeric Scoring
        critique_prompt = (
            f"Goal: {goal}\n\n"
            f"Candidate Solution:\n{current}\n\n"
            "Score this solution strictly from 0.0 to 10.0 based on correctness, edge cases, and code clarity. "
            "Set passed=True only if score >= 9.0."
        )
        crit_res = llm.models.generate_content(
            model=model,
            contents=critique_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ScoredCritique,
            ),
        )
        evaluation = getattr(crit_res, "parsed", None) or ScoredCritique.model_validate_json(
            crit_res.text
        )

        print(f"--- Round {round_num} Evaluation ---")
        print(f"Score: {evaluation.score}/10.0 | Passed: {evaluation.passed}")
        print(f"Feedback: {evaluation.feedback}\n")

        # Keep-the-best-so-far policy: retain candidate with highest numeric score
        if evaluation.score > best_score:
            best_score = evaluation.score
            best_solution = current
            print(f"⭐ Updated best-so-far solution (New Best Score: {best_score}/10.0)\n")

        # Exit if target score reached or max rounds cap reached
        if evaluation.passed or evaluation.score >= score_threshold:
            print(f"Quality threshold reached in round {round_num}!")
            break
        if round_num == max_round:
            print(f"Reached round cap ({max_round}).")
            break

        # 3. Revise based on critique
        revise_prompt = (
            f"Goal: {goal}\n\n"
            f"Current Draft (Score {evaluation.score}/10.0):\n{current}\n\n"
            f"Critique Feedback:\n{evaluation.feedback}\n\n"
            "Revise and improve the solution to address all critique points."
        )
        rev_res = llm.models.generate_content(
            model=model,
            contents=revise_prompt,
        )
        current = rev_res.text or ""
        print(f"--- Round {round_num + 1}: Revise ---")
        print(f"Draft {round_num + 1}:\n{current}\n")

    return best_solution, best_score

In [9]:
# Task demonstrating numeric scoring, keep-the-best-so-far policy, and reflection
task = (
    "Write a concise Python function `parse_query_params(url: str) -> dict[str, list[str]]` "
    "that parses query parameters into a dictionary mapping keys to lists of values, "
    "handling URL decoding, parameters without values, empty queries, and fragment stripping."
)

best_code, best_score = reflection_with_best_so_far(
    goal=task,
    llm=google_client,
    max_round=3,
)

print(f"=== Final Best-So-Far Solution (Score: {best_score}/10.0) ===")
print(best_code)

--- Round 1: Generate ---
Draft 1:
Here is the best and most concise complete solution using Python's standard library `urllib.parse` module. 

It correctly strips the fragment, parses percent-encoded keys and values, and preserves multiple values for the same parameter in a list (using `urllib.parse.parse_qs`, which naturally outputs `dict[str, list[str]]`).

```python
from urllib.parse import parse_qs, urlparse


def parse_query_params(url: str) -> dict[str, list[str]]:
    """Parses query parameters from a URL into a dictionary mapping keys to lists of values.

    Handles URL decoding, parameters without values, empty queries, and fragment stripping.
    """
    # urlparse strips the fragment (params.fragment) automatically when parsing
    parsed = urlparse(url)
    # parse_qs handles URL decoding and returns list of values per key
    return parse_qs(parsed.query, keep_blank_values=True)
```

### Example Usage:

```python
url = "https://example.com/path?name=Alice&hobby=reading&h

### Task 5

In [ ]:
class IssueDetail(BaseModel):
    """Specific issue requiring exact quoted evidence from the draft."""

    quoted_evidence: str = Field(
        ...,
        description="Exact verbatim substring quote from the draft demonstrating the flaw",
    )
    explanation: str = Field(
        ...,
        description="Explanation of why this quoted snippet is an issue or bug",
    )


class EvidenceCriticResult(BaseModel):
    """Critic evaluation permitting 'no issues' and requiring quoted evidence."""

    has_issues: bool = Field(
        ...,
        description="False if the draft has NO issues, True if issues exist",
    )
    issues: list[IssueDetail] = Field(
        default_factory=list,
        description="List of issues with verbatim quoted evidence, or empty if no issues",
    )
    verdict: str = Field(
        ...,
        description="Explicit verdict: 'NO ISSUES' if acceptable, otherwise 'NEEDS REVISION'",
    )


def evaluate_draft_with_evidence(
    goal: str,
    draft: str,
    llm: Client,
    model: str = "gemini-3.5-flash-lite",
) -> EvidenceCriticResult:
    """Critic prompt requiring quoted evidence for flaws and explicitly permitting 'no issues'."""
    prompt = f"""You are a precise, objective code reviewer evaluating a draft against the specified goal.

GOAL:
{goal}

DRAFT TO REVIEW:
{draft}

CRITIQUE RULES:
1. Grounding via Quoted Evidence: For EVERY issue found, you MUST provide an exact, verbatim quote from the draft in `quoted_evidence`. Do NOT invent or paraphrase snippets.
2. Permit 'No Issues': If the draft correctly satisfies the goal without logic errors or crashes, you MUST return `has_issues=False`, `issues=[]`, and `verdict='NO ISSUES'`. Do NOT invent nitpicks or hallucinate problems on good code.
3. If genuine flaws exist, set `has_issues=True`, provide quoted evidence for each, and set `verdict='NEEDS REVISION'`.
"""
    response = llm.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=EvidenceCriticResult,
        ),
    )
    return getattr(response, "parsed", None) or EvidenceCriticResult.model_validate_json(
        response.text
    )

In [13]:
# Test the critic on 5 seeded flawed drafts and 5 good drafts
goal_task = "Implement a function `def calculate_average(nums: list[float]) -> float | None` to safely return the average of a list of numbers, or None if empty."

flawed_drafts = [
    # 1. ZeroDivisionError on empty list
    "def calculate_average(nums):\n    return sum(nums) / len(nums)",
    # 2. Integer division truncating float values
    "def calculate_average(nums):\n    if not nums: return None\n    return sum(nums) // len(nums)",
    # 3. Off-by-one error in denominator
    "def calculate_average(nums):\n    if not nums: return None\n    return sum(nums) / (len(nums) - 1)",
    # 4. Modifying input list and returning wrong type
    "def calculate_average(nums):\n    if not nums: return None\n    nums.clear()\n    return 'done'",
    # 5. Typo in variable name (NameError)
    "def calculate_average(nums):\n    if not nums: return None\n    total = sum(nums)\n    return totl / len(nums)",
]

good_drafts = [
    # 1. Clean guard clause
    "def calculate_average(nums: list[float]) -> float | None:\n    if not nums:\n        return None\n    return sum(nums) / len(nums)",
    # 2. Explicit length check
    "def calculate_average(nums: list[float]) -> float | None:\n    if len(nums) == 0:\n        return None\n    return float(sum(nums)) / len(nums)",
    # 3. Defensive float check and docstring
    'def calculate_average(nums: list[float]) -> float | None:\n    """Return arithmetic mean or None if list is empty."""\n    if not nums:\n        return None\n    return sum(nums) / float(len(nums))',
    # 4. Iterative accumulator
    "def calculate_average(nums: list[float]) -> float | None:\n    if not nums:\n        return None\n    total = 0.0\n    for n in nums:\n        total += float(n)\n    return total / len(nums)",
    # 5. Using built-in statistics module safely
    "from statistics import mean, StatisticsError\ndef calculate_average(nums: list[float]) -> float | None:\n    try:\n        return float(mean(nums))\n    except StatisticsError:\n        return None",
]

print("=== Evaluating 5 Flawed Drafts (Expect has_issues=True with quoted evidence) ===")
flawed_correct = 0
for idx, draft in enumerate(flawed_drafts, 1):
    result = evaluate_draft_with_evidence(goal_task, draft, google_client)
    has_expected = result.has_issues is True
    flawed_correct += int(has_expected)
    print(f"\nFlawed Draft #{idx}: Verdict: {result.verdict} | Issues: {len(result.issues)}")
    for issue in result.issues:
        print(f"  - Quoted Evidence: {issue.quoted_evidence!r}")
        print(f"    Explanation: {issue.explanation}")

print("\n=== Evaluating 5 Good Drafts (Expect has_issues=False / NO ISSUES) ===")
good_correct = 0
for idx, draft in enumerate(good_drafts, 1):
    result = evaluate_draft_with_evidence(goal_task, draft, google_client)
    has_expected = result.has_issues is False
    good_correct += int(has_expected)
    print(
        f"Good Draft #{idx}: Verdict: {result.verdict} | has_issues={result.has_issues} | Issues: {result.issues}"
    )

print("\nSummary Accuracy:")
print(f"Flawed Detection: {flawed_correct}/5 (True Positives)")
print(f"Good Acceptance:  {good_correct}/5 (True Negatives, no false alarms)")
print(f"Overall Accuracy: {(flawed_correct + good_correct) / 10 * 100:.1f}%")

=== Evaluating 5 Flawed Drafts (Expect has_issues=True with quoted evidence) ===

Flawed Draft #1: Verdict: NEEDS REVISION | Issues: 1
  - Quoted Evidence: 'return sum(nums) / len(nums)'
    Explanation: The function does not handle empty lists. If nums is empty, len(nums) is 0, causing a ZeroDivisionError instead of returning None as specified in the goal.

Flawed Draft #2: Verdict: NEEDS REVISION | Issues: 1
  - Quoted Evidence: 'return sum(nums) // len(nums)'
    Explanation: The draft uses integer division (//) instead of standard division (/), which truncates the decimal part and causes the function to return an incorrect average for floating-point or non-evenly divisible numbers.

Flawed Draft #3: Verdict: NEEDS REVISION | Issues: 1
  - Quoted Evidence: 'return sum(nums) / (len(nums) - 1)'
    Explanation: The function calculates the average by dividing by (len(nums) - 1) instead of len(nums), which computes a sample variance denominator or incorrect mean instead of the standard 

### Task 6

### Task 7

### Task 8